# 09 Insight Text Template Refinement

Notebook ini digunakan untuk menerapkan deterministic template mapping pada teks insight final. Proses ini hanya mengubah teks display insight dan tidak mengubah relasi utama dataset.

## 1. Setup Path

In [6]:
from pathlib import Path
import pandas as pd

ROOT_DIR = Path('../../..').resolve()
INPUT_PATH = ROOT_DIR / 'data/processed/insights_clean.csv'
MAPPING_PATH = ROOT_DIR / 'data/mapping/insight_text_template_mapping.csv'
OUTPUT_PATH = ROOT_DIR / 'data/processed/insights_final.csv'
REPORT_PATH = ROOT_DIR / 'outputs/reports/insight_text_refinement_validation.md'

INPUT_PATH, MAPPING_PATH, OUTPUT_PATH, REPORT_PATH

(WindowsPath('C:/Data Codingan/student_stress_data_science/data/processed/insights_clean.csv'),
 WindowsPath('C:/Data Codingan/student_stress_data_science/data/mapping/insight_text_template_mapping.csv'),
 WindowsPath('C:/Data Codingan/student_stress_data_science/data/processed/insights_final.csv'),
 WindowsPath('C:/Data Codingan/student_stress_data_science/outputs/reports/insight_text_refinement_validation.md'))

## 2. Load Dataset dan Mapping

In [7]:
insights_df = pd.read_csv(INPUT_PATH)
mapping_df = pd.read_csv(MAPPING_PATH)

print('Insights shape:', insights_df.shape)
print('Mapping shape:', mapping_df.shape)
display(mapping_df.head())

Insights shape: (29551, 7)
Mapping shape: (78, 8)


,template_code,period_type,stress_level_hint,dominant_factor_hint,original_insight_text,revised_insight_text,revision_method,mapping_note
0,WEEKLY_WEEKLY_STABLE_NO_DOMINANT_FACTOR,weekly,weekly_stable,NaN,Pola stress mingguan relatif stabil. Tetap pan...,Pola stres mingguan terlihat relatif stabil. K...,deterministic_template_mapping_v1,Template formal tanpa subjek personal langsung.
1,WEEKLY_WEEKLY_HIGH_DAYS_NO_DOMINANT_FACTOR,weekly,weekly_high_days,NaN,"Selama 7 hari terakhir, terdapat beberapa hari...","Dalam tujuh hari terakhir, terdapat beberapa h...",deterministic_template_mapping_v1,Template formal tanpa subjek personal langsung.
2,DAILY_RENDAH_NO_DOMINANT_FACTOR,daily,rendah,NaN,"Stress score hari ini rendah. Pola tidur, mood...",Tingkat stres hari ini berada pada kategori re...,deterministic_template_mapping_v1,Template formal tanpa subjek personal langsung.
3,DAILY_SEDANG_ACADEMIC_PRESSURE_FINANCIAL_WORRY,daily,sedang,Academic Pressure + Financial Worry,Stress score hari ini sedang. Faktor yang perl...,Tingkat stres hari ini berada pada kategori se...,deterministic_template_mapping_v1,Template formal tanpa subjek personal langsung.
4,DAILY_SEDANG_ACADEMIC_PRESSURE_HEALTH_ISSUE,daily,sedang,Academic Pressure + Health Issue,Stress score hari ini sedang. Faktor yang perl...,Tingkat stres hari ini berada pada kategori se...,deterministic_template_mapping_v1,Template formal tanpa subjek personal langsung.


## 3. Validasi Mapping

In [8]:
required_input_columns = {'id', 'period_type', 'insight_text'}
required_mapping_columns = {'template_code', 'original_insight_text', 'revised_insight_text'}

missing_input = required_input_columns - set(insights_df.columns)
missing_mapping = required_mapping_columns - set(mapping_df.columns)

assert not missing_input, f'Kolom input tidak lengkap: {missing_input}'
assert not missing_mapping, f'Kolom mapping tidak lengkap: {missing_mapping}'
assert not mapping_df['original_insight_text'].duplicated().any(), 'Terdapat original_insight_text duplikat pada mapping.'

original_texts = set(insights_df['insight_text'].dropna())
mapped_texts = set(mapping_df['original_insight_text'].dropna())
unmapped_texts = sorted(original_texts - mapped_texts)

print('Unique insight_text pada dataset:', len(original_texts))
print('Jumlah mapping:', len(mapping_df))
print('Jumlah insight_text belum termapping:', len(unmapped_texts))
assert len(unmapped_texts) == 0, unmapped_texts[:10]

Unique insight_text pada dataset: 78
Jumlah mapping: 78
Jumlah insight_text belum termapping: 0


## 4. Validasi Larangan Kata Subjektif

In [9]:
forbidden_words = ['lo', 'gua', 'gue', 'elo', 'kamu', 'anda', 'pengguna']
violations = []
for text in mapping_df['revised_insight_text'].dropna().astype(str).unique():
    tokens = text.lower().replace(',', ' ').replace('.', ' ').replace(';', ' ').split()
    for word in forbidden_words:
        if word in tokens:
            violations.append((word, text))

print('Jumlah pelanggaran kata subjektif:', len(violations))
assert len(violations) == 0, violations[:10]

Jumlah pelanggaran kata subjektif: 0


## 5. Apply Mapping dan Export Dataset Final

In [10]:
revised_map = dict(zip(mapping_df['original_insight_text'], mapping_df['revised_insight_text']))
code_map = dict(zip(mapping_df['original_insight_text'], mapping_df['template_code']))

final_df = insights_df.copy()
final_df['insight_text_original'] = final_df['insight_text']
final_df['insight_template_code'] = final_df['insight_text'].map(code_map)
final_df['insight_text'] = final_df['insight_text'].map(revised_map)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(OUTPUT_PATH, index=False)

print('Output saved:', OUTPUT_PATH)
print('Final shape:', final_df.shape)
display(final_df.head())

Output saved: C:\Data Codingan\student_stress_data_science\data\processed\insights_final.csv
Final shape: (29551, 9)


,id,user_id,stress_prediction_id,weekly_summary_id,period_type,insight_text,created_at,insight_text_original,insight_template_code
0,1,1,1.0,NaN,daily,Tingkat stres hari ini berada pada kategori se...,2026-01-02 00:01:00,Stress score hari ini sedang. Faktor yang perl...,DAILY_SEDANG_LOW_MOOD_LOW_PHYSICAL_ACTIVITY
1,2,1,2.0,NaN,daily,Tingkat stres hari ini berada pada kategori se...,2026-01-02 20:46:00,Stress score hari ini sedang. Faktor yang perl...,DAILY_SEDANG_ACADEMIC_PRESSURE_HEALTH_ISSUE
2,3,1,3.0,NaN,daily,Tingkat stres hari ini berada pada kategori se...,2026-01-03 20:52:00,Stress score hari ini sedang. Faktor yang perl...,DAILY_SEDANG_HIGH_SCREEN_TIME
3,4,1,4.0,NaN,daily,Tingkat stres hari ini berada pada kategori se...,2026-01-05 00:59:00,Stress score hari ini sedang. Faktor yang perl...,DAILY_SEDANG_LOW_MOOD_HIGH_SCREEN_TIME
4,5,1,5.0,NaN,daily,Tingkat stres hari ini berada pada kategori se...,2026-01-05 23:12:00,Stress score hari ini sedang. Faktor yang perl...,DAILY_SEDANG_ACADEMIC_PRESSURE


## 6. Validation Report

In [11]:
checks = {
    'input_rows': len(insights_df),
    'output_rows': len(final_df),
    'unique_original_insight_text': insights_df['insight_text'].nunique(),
    'mapping_rows': len(mapping_df),
    'unique_revised_insight_text': mapping_df['revised_insight_text'].nunique(),
    'missing_revised_text': final_df['insight_text'].isna().sum(),
    'unmapped_original_text_count': len(unmapped_texts),
    'duplicate_id_count': final_df['id'].duplicated().sum(),
}

validation_table = pd.DataFrame(list(checks.items()), columns=['validation_item', 'value'])
display(validation_table)

assert checks['input_rows'] == checks['output_rows']
assert checks['missing_revised_text'] == 0
assert checks['unmapped_original_text_count'] == 0
assert checks['duplicate_id_count'] == 0

,validation_item,value
0,input_rows,29551
1,output_rows,29551
2,unique_original_insight_text,78
3,mapping_rows,78
4,unique_revised_insight_text,78
5,missing_revised_text,0
6,unmapped_original_text_count,0
7,duplicate_id_count,0
